## Iteration 7 — MMR Diversity-Aware Retrieval

### Objective

Iteration 6 achieved the best Any Top-3 = 83.3% and retained Any Top-5 = 91.7%, but the remaining failure showed a redundancy problem.
For the missed-test question, the FAISS Top-5 contained several very similar Special Consideration chunks:

- POL-01_63
- SC-01_8
- SC-01_6
- SC-01_9
- SC-01_7
  while the relevant EAA-02_43 was only rank 6.
  Iteration 7 tests Maximal Marginal Relevance (MMR) to balance:

1. Relevance to the question
2. Diversity among selected chunks
   Pipeline
   Original question → FAISS Top-10 → MMR selection → Top-5
   No query expansion, BM25, or reranker is used in this iteration.
   The document representation remains identical to Iteration 2:
   category + section + content


In [1]:
!pip -q install sentence-transformers faiss-cpu pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer
from IPython.display import display

print("Libraries loaded.")

c:\Users\KRUSHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded.


In [3]:
chunks_path = "./02_rmit_assessment_support_chunks.csv"
test_path = "./test_prompts.json"

chunks_df = pd.read_csv(chunks_path)

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_df = pd.DataFrame(test_data["tests"])

print("Chunks:", len(chunks_df))
print("Test prompts:", len(test_df))

Chunks: 71
Test prompts: 12


## 1. Recreate the same document representation as Iteration 2

No document-side changes are introduced.


In [4]:
chunks_df["embedding_text"] = (
    chunks_df["category"].fillna("") + ". " +
    chunks_df["section"].fillna("") + ". " +
    chunks_df["content"].fillna("")
)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    chunks_df["embedding_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

# Normalised embeddings allow cosine similarity using inner product
faiss.normalize_L2(embeddings)

faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)

print("FAISS vectors:", faiss_index.ntotal)
print("Embedding dimension:", embeddings.shape[1])

Batches: 100%|██████████| 3/3 [00:01<00:00,  2.98it/s]

FAISS vectors: 71
Embedding dimension: 384


## 2. Retrieve the FAISS candidate pool

MMR needs a larger candidate pool than the final Top-5. We retrieve Top-10 and then select five diverse candidates.


In [5]:
def faiss_candidates(question, k=10):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(
        query_embedding,
        k
    )

    return query_embedding[0], [
        {
            "index": int(idx),
            "faiss_rank": rank + 1,
            "faiss_score": float(score)
        }
        for rank, (idx, score) in enumerate(
            zip(indices[0], scores[0])
        )
    ]

## 3. MMR selection

MMR uses:
MMR = λ × relevance − (1 − λ) × maximum similarity to already selected chunks
With lambda = 0.70, relevance remains the stronger signal while allowing diversity to influence selection.


In [6]:
def mmr_select(
    question,
    candidate_k=10,
    final_k=5,
    lambda_param=0.70
):
    query_embedding, candidates = faiss_candidates(
        question,
        k=candidate_k
    )

    candidate_indices = [
        item["index"]
        for item in candidates
    ]

    candidate_embeddings = embeddings[
        candidate_indices
    ]

    # Query-to-document similarity
    query_embedding_2d = query_embedding.reshape(1, -1)

    relevance_scores = np.dot(
        candidate_embeddings,
        query_embedding_2d.T
    ).flatten()

    selected_positions = []
    remaining_positions = list(
        range(len(candidate_indices))
    )

    mmr_scores_history = []

    while remaining_positions and len(selected_positions) < final_k:

        best_position = None
        best_mmr = -np.inf

        for position in remaining_positions:

            relevance = relevance_scores[position]

            if not selected_positions:
                redundancy = 0.0
            else:
                selected_embeddings = candidate_embeddings[
                    selected_positions
                ]

                similarities = np.dot(
                    selected_embeddings,
                    candidate_embeddings[position]
                )

                redundancy = float(np.max(similarities))

            mmr_score = (
                lambda_param * relevance
                - (1 - lambda_param) * redundancy
            )

            if mmr_score > best_mmr:
                best_mmr = mmr_score
                best_position = position

        selected_positions.append(best_position)
        remaining_positions.remove(best_position)

        mmr_scores_history.append({
            "candidate_position": best_position,
            "mmr_score": best_mmr
        })

    rows = []

    for final_rank, position in enumerate(
        selected_positions,
        start=1
    ):
        idx = candidate_indices[position]
        row = chunks_df.iloc[idx]

        original_candidate = candidates[position]

        rows.append({
            "final_rank": final_rank,
            "index": idx,
            "chunk_id": row["chunk_id"],
            "source_id": row["source_id"],
            "category": row["category"],
            "section": row["section"],
            "content": row["content"],
            "faiss_rank": original_candidate["faiss_rank"],
            "faiss_score": original_candidate["faiss_score"],
            "mmr_score": mmr_scores_history[
                final_rank - 1
            ]["mmr_score"]
        })

    return pd.DataFrame(rows)

## 4. Test the previously difficult missed-test question

This is the key diagnostic for Iteration 7.


In [7]:
q = (
    "I missed a test because I was unwell, "
    "How long do I have to apply for special consideration?"
)

expected = ["SC-01_8", "EAA-02_43"]

mmr_results = mmr_select(
    q,
    candidate_k=10,
    final_k=5,
    lambda_param=0.70
)

print("Expected:", expected)

display(
    mmr_results[
        [
            "final_rank",
            "chunk_id",
            "source_id",
            "faiss_rank",
            "faiss_score",
            "mmr_score"
        ]
    ]
)

Expected: ['SC-01_8', 'EAA-02_43']


,final_rank,chunk_id,source_id,faiss_rank,faiss_score,mmr_score
0,1,SC-01_8,SC-01,1,0.595475,0.416833
1,2,SC-01_7,SC-01,9,0.457805,0.208737
2,3,POL-01_63,POL-01,2,0.571715,0.206915
3,4,EXT-01_0,EXT-01,3,0.493480,0.178756
4,5,EAA-02_43,EAA-02,6,0.478629,0.147871


## 5. Compare MMR settings on the difficult question

A small lambda sensitivity check helps determine whether the result depends strongly on the chosen diversity weight.


In [8]:
lambda_results = []

for lambda_value in [0.50, 0.60, 0.70, 0.80, 0.90]:

    result = mmr_select(
        q,
        candidate_k=10,
        final_k=5,
        lambda_param=lambda_value
    )

    retrieved = result["chunk_id"].tolist()

    lambda_results.append({
        "lambda": lambda_value,
        "top5": retrieved,
        "SC-01_8_in_top5": "SC-01_8" in retrieved,
        "EAA-02_43_in_top5": "EAA-02_43" in retrieved
    })

lambda_df = pd.DataFrame(lambda_results)

display(lambda_df)

,lambda,top5,SC-01_8_in_top5,EAA-02_43_in_top5
0,0.5,"[SC-01_8, SC-01_7, EAA-02_43, POL-01_63, POL-0...",True,True
1,0.6,"[SC-01_8, SC-01_7, EAA-02_43, POL-01_63, POL-0...",True,True
2,0.7,"[SC-01_8, SC-01_7, POL-01_63, EXT-01_0, EAA-02...",True,True
3,0.8,"[SC-01_8, POL-01_63, EXT-01_0, EAA-02_43, SC-0...",True,True
4,0.9,"[SC-01_8, POL-01_63, EXT-01_0, SC-01_9, EAA-02...",True,True


## 6. Evaluation


In [9]:
def any_expected_retrieved(retrieved, expected):
    return any(chunk in retrieved for chunk in expected)


def all_expected_retrieved(retrieved, expected):
    return all(chunk in retrieved for chunk in expected)


def evaluate_iteration7(test_df, lambda_param=0.70):

    results = []

    for _, test in test_df.iterrows():

        question = test["prompt"]
        expected_chunks = test["chunks"]

        retrieved_df = mmr_select(
            question,
            candidate_k=10,
            final_k=5,
            lambda_param=lambda_param
        )

        retrieved_chunks = (
            retrieved_df["chunk_id"].tolist()
        )

        top1 = retrieved_chunks[:1]
        top3 = retrieved_chunks[:3]
        top5 = retrieved_chunks[:5]

        expected_ranks = [
            retrieved_chunks.index(chunk) + 1
            for chunk in expected_chunks
            if chunk in retrieved_chunks
        ]

        results.append({
            "prompt": question,
            "expected_chunks": expected_chunks,

            "top1_chunk": top1[0] if top1 else None,
            "top3_chunks": top3,
            "top5_chunks": top5,

            "expected_rank_in_top5":
                min(expected_ranks)
                if expected_ranks else None,

            "top1_hit":
                any_expected_retrieved(
                    top1,
                    expected_chunks
                ),

            "top3_hit":
                any_expected_retrieved(
                    top3,
                    expected_chunks
                ),

            "top5_hit":
                any_expected_retrieved(
                    top5,
                    expected_chunks
                ),

            "top1_all_hit":
                all_expected_retrieved(
                    top1,
                    expected_chunks
                ),

            "top3_all_hit":
                all_expected_retrieved(
                    top3,
                    expected_chunks
                ),

            "top5_all_hit":
                all_expected_retrieved(
                    top5,
                    expected_chunks
                )
        })

    return pd.DataFrame(results)

In [10]:
iteration7_df = evaluate_iteration7(
    test_df,
    lambda_param=0.70
)

print("ITERATION 7 — MMR RETRIEVAL")
print("---------------------------")

print("\nAny expected chunk retrieved:")
print(
    f"Top-1: {iteration7_df['top1_hit'].mean():.1%}"
)
print(
    f"Top-3: {iteration7_df['top3_hit'].mean():.1%}"
)
print(
    f"Top-5: {iteration7_df['top5_hit'].mean():.1%}"
)

print("\nAll expected chunks retrieved:")
print(
    f"Top-1: {iteration7_df['top1_all_hit'].mean():.1%}"
)
print(
    f"Top-3: {iteration7_df['top3_all_hit'].mean():.1%}"
)
print(
    f"Top-5: {iteration7_df['top5_all_hit'].mean():.1%}"
)

ITERATION 7 — MMR RETRIEVAL
---------------------------

Any expected chunk retrieved:
Top-1: 50.0%
Top-3: 83.3%
Top-5: 100.0%

All expected chunks retrieved:
Top-1: 41.7%
Top-3: 50.0%
Top-5: 83.3%


In [11]:
comparison_df = pd.DataFrame({
    "Metric": [
        "Any expected — Top-1",
        "Any expected — Top-3",
        "Any expected — Top-5",
        "All expected — Top-1",
        "All expected — Top-3",
        "All expected — Top-5"
    ],

    "Iteration 2": [
        0.500, 0.750, 0.917,
        0.417, 0.583, 0.750
    ],

    "Corrected Iteration 5": [
        0.583, 0.667, 0.917,
        0.500, 0.667, 0.750
    ],

    "Iteration 6": [
        0.500, 0.833, 0.917,
        0.417, 0.667, 0.750
    ],

    "Iteration 7": [
        iteration7_df["top1_hit"].mean(),
        iteration7_df["top3_hit"].mean(),
        iteration7_df["top5_hit"].mean(),
        iteration7_df["top1_all_hit"].mean(),
        iteration7_df["top3_all_hit"].mean(),
        iteration7_df["top5_all_hit"].mean()
    ]
})

comparison_df["I7 vs I2"] = (
    comparison_df["Iteration 7"]
    - comparison_df["Iteration 2"]
)

display(
    comparison_df.style.format({
        "Iteration 2": "{:.1%}",
        "Corrected Iteration 5": "{:.1%}",
        "Iteration 6": "{:.1%}",
        "Iteration 7": "{:.1%}",
        "I7 vs I2": "{:+.1%}"
    })
)

,Metric,Iteration 2,Corrected Iteration 5,Iteration 6,Iteration 7,I7 vs I2
0,Any expected — Top-1,50.0%,58.3%,50.0%,50.0%,+0.0%
1,Any expected — Top-3,75.0%,66.7%,83.3%,83.3%,+8.3%
2,Any expected — Top-5,91.7%,91.7%,91.7%,100.0%,+8.3%
3,All expected — Top-1,41.7%,50.0%,41.7%,41.7%,-0.0%
4,All expected — Top-3,58.3%,66.7%,66.7%,50.0%,-8.3%
5,All expected — Top-5,75.0%,75.0%,75.0%,83.3%,+8.3%


In [12]:
top5_failures_7 = iteration7_df[
    iteration7_df["top5_hit"] == False
][[
    "prompt",
    "expected_chunks",
    "top1_chunk",
    "top3_chunks",
    "top5_chunks",
    "expected_rank_in_top5"
]]

print("Iteration 7 Top-5 failures:", len(top5_failures_7))
display(top5_failures_7)

Iteration 7 Top-5 failures: 0


,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5


In [13]:
# Inspect the Top-5 for all test cases
display(
    iteration7_df[
        [
            "prompt",
            "expected_chunks",
            "top1_chunk",
            "top3_chunks",
            "top5_chunks",
            "expected_rank_in_top5"
        ]
    ]
)

,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5
0,I've got a timed quiz that I won't be able to ...,[EXT-01_1],EXT-01_1,"[EXT-01_1, EXT-01_0, EAA-02_38]","[EXT-01_1, EXT-01_0, EAA-02_38, EXT-01_2, POL-...",1
1,"I've recently lost my job, is this a valid rea...",[SC-01_7],SC-01_4,"[SC-01_4, SC-01_8, SC-01_13]","[SC-01_4, SC-01_8, SC-01_13, SC-01_7, POL-01_63]",4
2,So my hard drive with my assignment stopped wo...,[SC-01_16],EXT-01_1,"[EXT-01_1, SC-01_16, EAA-02_38]","[EXT-01_1, SC-01_16, EAA-02_38, SC-01_6, EXT-0...",2
3,I've just booked my first appointment with ELS...,[EAA-02_29],EAA-02_29,"[EAA-02_29, EAA-02_28, EAA-02_20]","[EAA-02_29, EAA-02_28, EAA-02_20, EAA-01_18, E...",1
4,I emailed my course coordinator three days ago...,[EAA-02_38],EAA-02_38,"[EAA-02_38, EXT-01_0, SC-01_9]","[EAA-02_38, EXT-01_0, SC-01_9, EAA-02_34, POL-...",1
5,Having a hard time reading through journal art...,[EAA-02_46],EAA-02_51,"[EAA-02_51, EAA-02_47, SC-01_15]","[EAA-02_51, EAA-02_47, SC-01_15, EAA-02_46, EA...",4
6,My mate got flagged for plagiarism because he ...,[POL-01_70],POL-01_70,"[POL-01_70, EXT-01_3, POL-01_56]","[POL-01_70, EXT-01_3, POL-01_56, EAA-02_19, PO...",1
7,Who actually sets RMIT's official academic cal...,[POL-01_67],POL-01_67,"[POL-01_67, EAA-02_50, EAA-02_52]","[POL-01_67, EAA-02_50, EAA-02_52, SC-01_11, EX...",1
8,"I missed a test because I was unwell, How long...","[SC-01_8, EAA-02_43]",EAA-01_18,"[EAA-01_18, POL-01_63, EAA-02_43]","[EAA-01_18, POL-01_63, EAA-02_43, SC-01_8, POL...",3
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]",EXT-01_0,"[EXT-01_0, EAA-02_41, EXT-01_1]","[EXT-01_0, EAA-02_41, EXT-01_1, POL-01_62, EAA...",3


In [14]:
iteration7_df.to_csv(
    "./10_iteration7_mmr_results.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison_df.to_csv(
    "./10_iteration2_to_iteration7_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

lambda_df.to_csv(
    "./iteration7_lambda_sensitivity.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Results saved.")

Results saved.
